In [ ]:
!pip install nba_api -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, time
import pandas as pd
from nba_api.stats.endpoints import leaguegamefinder

DATA_DIR = '/content/drive/MyDrive/nba-xgboost/data'
os.makedirs(DATA_DIR, exist_ok=True)

# 2016-17 is warm-up only; 2017-18 to 2025-26 is the modeling data
SEASONS = [f"{y}-{str(y+1)[-2:]}" for y in range(2016, 2026)]
print(SEASONS, len(SEASONS))

['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26'] 10


In [1]:
import os
DATA_DIR = '/content/drive/MyDrive/nba-xgboost/data'
print(sorted(os.listdir(DATA_DIR)))

['games_2016-17.csv', 'games_2017-18.csv', 'games_2018-19.csv', 'games_2019-20.csv', 'games_2020-21.csv', 'games_2021-22.csv', 'games_2022-23.csv', 'games_2023-24.csv', 'games_2024-25.csv', 'games_2025-26.csv']


In [2]:
import pandas as pd

SEASONS = [f"{y}-{str(y+1)[-2:]}" for y in range(2016, 2026)]
frames = [pd.read_csv(f"{DATA_DIR}/games_{s}.csv") for s in SEASONS]
raw = pd.concat(frames, ignore_index=True)
raw['GAME_DATE'] = pd.to_datetime(raw['GAME_DATE'])
raw.to_csv(f"{DATA_DIR}/games_raw_all.csv", index=False)
print(raw.shape)

(23958, 28)


In [3]:
print(raw.groupby('SEASON_ID').size())

SEASON_ID
22016    2460
22017    2460
22018    2460
22019    2118
22020    2160
22021    2460
22022    2460
22023    2460
22024    2460
22025    2460
dtype: int64


In [4]:
print(raw.groupby('GAME_ID').size().value_counts())


2    11979
Name: count, dtype: int64


In [5]:
needed = ['TEAM_ID','GAME_ID','GAME_DATE','MATCHUP','WL',
          'PTS','FG_PCT','FG3_PCT','FT_PCT','FGM','FGA','FG3M','FG3A',
          'FTM','FTA','AST','TOV','OREB','DREB','REB','STL','BLK','PF','PLUS_MINUS']
print("Missing columns:", [c for c in needed if c not in raw.columns])

Missing columns: []


In [6]:
print(raw['GAME_DATE'].min(), "->", raw['GAME_DATE'].max())
print(raw[needed].isnull().sum().sum(), "null values")
print(raw['WL'].value_counts(dropna=False))


2016-10-25 00:00:00 -> 2026-04-12 00:00:00
1 null values
WL
L    11979
W    11979
Name: count, dtype: int64


In [7]:
n = raw[raw['GAME_DATE'] >= '2017-10-01']['GAME_ID'].nunique()
print("Games from 2017-18 onward:", n, "| train ~", int(n*0.8), "| test ~", n-int(n*0.8))

Games from 2017-18 onward: 10749 | train ~ 8599 | test ~ 2150
